# Guardian AI — MuRIL Fine-Tuning Notebook (v2 — Fixed)

Fine-tunes `google/muril-base-cased` on the Guardian AI safety dataset.

**v2 Fixes:**
- Fixed LayerNorm key mismatch (gamma/beta → weight/bias) that corrupted the saved model
- Added post-save verification to confirm model works before downloading
- Improved training args (6 epochs, fp16, gradient accumulation)

**Setup:** Colab runtime via VS Code Colab extension (code runs on Google's GPU servers)

**Before running:**
1. Upload `train.csv` and `test.csv` to your Google Drive inside a folder called `GuardianAi/`
2. Connect to a Colab GPU runtime (T4 recommended)

**Output:** Fine-tuned model saved to Google Drive at `MyDrive/GuardianAi/muril_finetuned/`

## Step 1 — Install dependencies

In [ ]:
!pip install transformers datasets torch scikit-learn accelerate -q
print('All dependencies installed.')

## Step 2 — Mount Google Drive & configure paths
Your `train.csv` and `test.csv` should be inside `MyDrive/GuardianAi/`.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# ── Paths on Google Drive ──
DRIVE_PROJECT = '/content/drive/MyDrive/GuardianAi'

TRAIN_CSV = os.path.join(DRIVE_PROJECT, 'train.csv')
TEST_CSV  = os.path.join(DRIVE_PROJECT, 'test.csv')
SAVE_DIR  = os.path.join(DRIVE_PROJECT, 'muril_finetuned')

# Verify files exist
assert os.path.exists(TRAIN_CSV), f'train.csv not found! Upload it to Google Drive at: MyDrive/GuardianAi/train.csv'
assert os.path.exists(TEST_CSV),  f'test.csv not found! Upload it to Google Drive at: MyDrive/GuardianAi/test.csv'

print(f'Drive project : {DRIVE_PROJECT}')
print(f'Train CSV     : {TRAIN_CSV}')
print(f'Test CSV      : {TEST_CSV}')
print(f'Model output  : {SAVE_DIR}')
print()
print('Contents of GuardianAi folder:')
for f in os.listdir(DRIVE_PROJECT):
    print(f'  {f}')
print()
print('All paths verified.')

## Step 3 — Load & inspect data

In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Drop any rows with missing text or label
train_df = train_df.dropna(subset=['text', 'label'])
test_df  = test_df.dropna(subset=['text', 'label'])

print(f'Train rows : {len(train_df)}')
print(f'Test rows  : {len(test_df)}')
print()
print('Train label distribution:')
print(train_df['label'].value_counts().rename({0: 'SAFE', 1: 'EMERGENCY'}))
print()
print('Sample rows:')
train_df.sample(5)

## Step 4 — Tokenize

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'google/muril-base-cased'
MAX_LEN    = 128

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SafetyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['text'].tolist()
        self.labels    = df['label'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SafetyDataset(train_df, tokenizer, MAX_LEN)
eval_dataset  = SafetyDataset(test_df,  tokenizer, MAX_LEN)

print(f'Train dataset: {len(train_dataset)} samples')
print(f'Eval  dataset: {len(eval_dataset)} samples')
print('Tokenization complete.')

## Step 5 — Load MuRIL model + Fix LayerNorm keys

**CRITICAL FIX:** MuRIL uses old TensorFlow-style LayerNorm naming (`gamma`/`beta`)
instead of PyTorch convention (`weight`/`bias`). This mismatch causes the Trainer's
`load_best_model_at_end` to silently corrupt all LayerNorm weights when reloading checkpoints.

We fix this by renaming the parameters right after loading.

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'SAFE', 1: 'EMERGENCY'},
    label2id={'SAFE': 0, 'EMERGENCY': 1},
)

# ══════════════════════════════════════════════════════════════════
#  CRITICAL FIX: Rename LayerNorm parameters gamma/beta → weight/bias
#  Without this, the Trainer's checkpoint reload silently corrupts
#  all LayerNorm weights, producing a model that outputs ~50/50.
# ══════════════════════════════════════════════════════════════════
fixed_count = 0
for module in model.modules():
    if hasattr(module, 'gamma') and isinstance(module.gamma, torch.nn.Parameter):
        module._parameters['weight'] = module._parameters.pop('gamma')
        fixed_count += 1
    if hasattr(module, 'beta') and isinstance(module.beta, torch.nn.Parameter):
        module._parameters['bias'] = module._parameters.pop('beta')
        fixed_count += 1

print(f'  Fixed {fixed_count} LayerNorm parameters (gamma/beta -> weight/bias)')

# Verify the fix worked
old_keys = [k for k in model.state_dict().keys() if '.gamma' in k or '.beta' in k]
assert len(old_keys) == 0, f'Fix failed! Still have old keys: {old_keys[:3]}...'
print('  All LayerNorm keys now use standard naming.')

model.to(device)
print('Model loaded and moved to', device)

## Step 6 — Fine-tune with HuggingFace Trainer

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy':         accuracy_score(labels, preds),
        'f1_emergency':     f1_score(labels, preds, pos_label=1),
        'recall_emergency': recall_score(labels, preds, pos_label=1),
    }

training_args = TrainingArguments(
    output_dir                  = './muril_finetuned',
    num_train_epochs            = 6,
    per_device_train_batch_size  = 16,
    per_device_eval_batch_size   = 32,
    gradient_accumulation_steps  = 2,
    warmup_steps                = 50,
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    fp16                        = True,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_emergency',
    greater_is_better           = True,
    logging_steps               = 20,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    compute_metrics = compute_metrics,
)

print('Starting fine-tuning ...')
print('This takes ~10-20 minutes on a T4 GPU.')
print()
trainer.train()

## Step 7 — Final evaluation on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

results = trainer.evaluate()
print('\n=== Final Eval Results ===')
for k, v in results.items():
    print(f'  {k:<35} {v:.4f}')

# Detailed report
preds_output = trainer.predict(eval_dataset)
preds  = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print()
print('=== Classification Report ===')
print(classification_report(labels, preds, target_names=['SAFE', 'EMERGENCY']))

cm = confusion_matrix(labels, preds)
print('=== Confusion Matrix (rows=actual, cols=predicted) ===')
print(f'               Pred SAFE   Pred EMERGENCY')
print(f'  Actual SAFE      {cm[0][0]:>5}          {cm[0][1]:>5}')
print(f'  Actual EMER      {cm[1][0]:>5}          {cm[1][1]:>5}')

## Step 8 — Save model to Google Drive

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print()
print(f'Model saved to Google Drive: {SAVE_DIR}')
print(f'Files saved:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f'  {f} ({size / 1024 / 1024:.1f} MB)' if size > 1024*1024 else f'  {f} ({size} bytes)')

## Step 9 — Verify saved model actually works

**NEW:** This step loads the model back from disk and tests it on sample sentences.
If you see correct predictions here, the model is safe to download.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print('=== Verifying saved model ===')
print(f'Loading from: {SAVE_DIR}')
print()

# Load fresh from disk (simulates what your local code will do)
verify_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
verify_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
verify_model.eval()

# Check for old-style keys in saved model
old_keys = [k for k in verify_model.state_dict().keys() if '.gamma' in k or '.beta' in k]
if old_keys:
    print(f'WARNING: Saved model still has {len(old_keys)} old-style keys!')
else:
    print('Key naming: All LayerNorm keys correct (weight/bias)')

# Test on sample sentences
test_sentences = [
    ('the weather is really nice today', 'SAFE'),
    ('someone is following me please help', 'EMERGENCY'),
    ('I love this coffee', 'SAFE'),
    ('I am gonna kill you', 'EMERGENCY'),
    ('What should we have for dinner', 'SAFE'),
    ('He locked the door I cannot leave', 'EMERGENCY'),
]

print()
print(f'{"Text":<45} {"Expected":<12} {"Predicted":<12} {"Emergency%":<10} {"Status"}')
print('-' * 95)

all_correct = True
for text, expected in test_sentences:
    inputs = verify_tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = verify_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    emergency_prob = probs[0][1].item()
    predicted = 'EMERGENCY' if emergency_prob > 0.5 else 'SAFE'
    correct = predicted == expected
    all_correct = all_correct and correct
    status = 'PASS' if correct else 'FAIL'
    print(f'{text:<45} {expected:<12} {predicted:<12} {emergency_prob:<10.1%} {status}')

print()
if all_correct:
    print('ALL TESTS PASSED — Model is working correctly!')
    print()
    print('Next steps:')
    print('  1. Download the muril_finetuned/ folder from Google Drive')
    print('  2. Put it in your GuardianAi/models/ folder as fine-tuned-v1')
    print('  3. Run: python evaluate.py --model ./models/fine-tuned-v1')
else:
    print('SOME TESTS FAILED — Something is still wrong with the model.')
    print('Check the training logs above for errors.')